In [1]:
# Import libraries required for data preprocessing

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [2]:
# Load the original EMI prediction dataset

df = pd.read_csv("../data/emi_prediction_dataset.csv")

# Display dataset shape
print("Dataset Shape:", df.shape)

Dataset Shape: (404800, 27)


C:\Users\kotgi\AppData\Local\Temp\ipykernel_25640\4143280056.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/emi_prediction_dataset.csv")


In [4]:
# Create a copy of the dataset for preprocessing

data = df.copy()

# Display the first 5 rows
data.head()

,age,gender,marital_status,education,monthly_salary,employment_type,years_of_employment,company_type,house_type,monthly_rent,...,existing_loans,current_emi_amount,credit_score,bank_balance,emergency_fund,emi_scenario,requested_amount,requested_tenure,emi_eligibility,max_monthly_emi
0,38.0,Female,Married,Professional,82600.0,Private,0.9,Mid-size,Rented,20000.0,...,Yes,23700.0,660.0,303200.0,70200.0,Personal Loan EMI,850000.0,15,Not_Eligible,500.0
1,38.0,Female,Married,Graduate,21500.0,Private,7.0,MNC,Family,0.0,...,Yes,4100.0,714.0,92500.0,26900.0,E-commerce Shopping EMI,128000.0,19,Not_Eligible,700.0
2,38.0,Male,Married,Professional,86100.0,Private,5.8,Startup,Own,0.0,...,No,0.0,650.0,672100.0,324200.0,Education EMI,306000.0,16,Eligible,27775.0
3,58.0,Female,Married,High School,66800.0,Private,2.2,Mid-size,Own,0.0,...,No,0.0,685.0,440900.0,178100.0,Vehicle EMI,304000.0,83,Eligible,16170.0
4,48.0,Female,Married,Professional,57300.0,Private,3.4,Mid-size,Family,0.0,...,No,0.0,770.0,97300.0,28200.0,Home Appliances EMI,252000.0,7,Not_Eligible,500.0


In [5]:
# Define the target variables for our two machine learning tasks

classification_target = 'emi_eligibility'
regression_target = 'max_monthly_emi'

print("Classification Target:", classification_target)
print("Regression Target:", regression_target)

Classification Target: emi_eligibility
Regression Target: max_monthly_emi


In [6]:
# Create the feature dataset by removing both target columns

feature_columns = [
    column for column in data.columns
    if column not in [classification_target, regression_target]
]

X = data[feature_columns]

print("Number of Features:", len(feature_columns))
print("\nFeature Columns:")
print(feature_columns)

Number of Features: 25

Feature Columns:
['age', 'gender', 'marital_status', 'education', 'monthly_salary', 'employment_type', 'years_of_employment', 'company_type', 'house_type', 'monthly_rent', 'family_size', 'dependents', 'school_fees', 'college_fees', 'travel_expenses', 'groceries_utilities', 'other_monthly_expenses', 'existing_loans', 'current_emi_amount', 'credit_score', 'bank_balance', 'emergency_fund', 'emi_scenario', 'requested_amount', 'requested_tenure']


In [7]:
# Separate classification and regression target variables

y_classification = data[classification_target]
y_regression = data[regression_target]

print("Classification Target Shape:", y_classification.shape)
print("Regression Target Shape:", y_regression.shape)

Classification Target Shape: (404800,)
Regression Target Shape: (404800,)


In [8]:
# Check the data types of all input features

X.dtypes

age                        object
gender                     object
marital_status             object
education                  object
monthly_salary             object
employment_type            object
years_of_employment       float64
company_type               object
house_type                 object
monthly_rent              float64
family_size                 int64
dependents                  int64
school_fees               float64
college_fees              float64
travel_expenses           float64
groceries_utilities       float64
other_monthly_expenses    float64
existing_loans             object
current_emi_amount        float64
credit_score              float64
bank_balance               object
emergency_fund            float64
emi_scenario               object
requested_amount          float64
requested_tenure            int64
dtype: object

In [9]:
# Identify all columns currently stored as object

object_columns = X.select_dtypes(include='object').columns.tolist()

print("Object Columns:")
print(object_columns)

Object Columns:
['age', 'gender', 'marital_status', 'education', 'monthly_salary', 'employment_type', 'company_type', 'house_type', 'existing_loans', 'bank_balance', 'emi_scenario']


In [10]:
# Inspect sample unique values in columns that may contain numerical data

columns_to_inspect = [
    'age',
    'monthly_salary',
    'existing_loans',
    'bank_balance'
]

for column in columns_to_inspect:
    print(f"\n--- {column} ---")
    print(X[column].dropna().unique()[:20])


--- age ---
[38.0 58.0 48.0 32.0 27.0 47.0 37.0 31.0 59.0 49.0 33.0 26.0 39.0 57.0
 28.0 '58' '38' '48' '32' '27']

--- monthly_salary ---
['82600.0' '21500.0' '86100.0' '66800.0' '57300.0' '38800.0' '27100.0'
 '392044.0' '47700.0' '129200.0' '58600.0' '47000.0' '11837.0' '53900.0'
 '110800.0' '64000.0' '46600.0' '77400.0' '40700.0' '47500.0']

--- existing_loans ---
['Yes' 'No']

--- bank_balance ---
['303200.0' '92500.0' '672100.0' '440900.0' '97300.0' '260800.0' '68000.0'
 '184600.0' '235600.0' '963200.0' '300700.0' '185100.0' '129600.0'
 '350500.0' '157300.0' '38700.0' '125700.0' '551200.0' '118200.0'
 '134300.0']


In [11]:
# Check how many values in the suspicious columns are non-numeric

for column in columns_to_inspect:
    numeric_values = pd.to_numeric(X[column], errors='coerce')
    invalid_count = numeric_values.isna().sum() - X[column].isna().sum()

    print(f"{column}: {invalid_count} non-numeric values")

age: 3 non-numeric values
monthly_salary: 1993 non-numeric values
existing_loans: 404800 non-numeric values
bank_balance: 1966 non-numeric values


In [12]:
# Convert numeric columns to numeric format
numeric_columns = ['age', 'monthly_salary', 'bank_balance']

for column in numeric_columns:
    X[column] = pd.to_numeric(X[column], errors='coerce')

print("Numeric columns converted successfully.")
print(X[numeric_columns].dtypes)

Numeric columns converted successfully.
age               float64
monthly_salary    float64
bank_balance      float64
dtype: object


C:\Users\kotgi\AppData\Local\Temp\ipykernel_25640\2876532109.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[column] = pd.to_numeric(X[column], errors='coerce')


In [13]:
# Check missing values after numeric conversion

print("Missing values in numeric columns:")
print(X[numeric_columns].isnull().sum())

Missing values in numeric columns:
age                  3
monthly_salary    1993
bank_balance      4392
dtype: int64


In [14]:
# Fill missing values in numeric columns with the median

for column in numeric_columns:
    X[column] = X[column].fillna(X[column].median())

print("Missing values filled successfully.")
print(X[numeric_columns].isnull().sum())

Missing values filled successfully.
age               0
monthly_salary    0
bank_balance      0
dtype: int64


C:\Users\kotgi\AppData\Local\Temp\ipykernel_25640\2344852464.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[column] = X[column].fillna(X[column].median())
C:\Users\kotgi\AppData\Local\Temp\ipykernel_25640\2344852464.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[column] = X[column].fillna(X[column].median())
C:\Users\kotgi\AppData\Local\Temp\ipykernel_25640\2344852464.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,co

In [15]:
# Convert existing_loans from Yes/No to 1/0

X['existing_loans'] = X['existing_loans'].map({
    'Yes': 1,
    'No': 0
})

print("existing_loans encoded successfully.")
print(X['existing_loans'].value_counts())

existing_loans encoded successfully.
existing_loans
0    243227
1    161573
Name: count, dtype: int64


C:\Users\kotgi\AppData\Local\Temp\ipykernel_25640\3587499594.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['existing_loans'] = X['existing_loans'].map({


In [16]:
# Check unique values in categorical columns

categorical_columns = X.select_dtypes(include='object').columns

print("Categorical columns:")
print(list(categorical_columns))

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(X[column].value_counts(dropna=False))

Categorical columns:
['gender', 'marital_status', 'education', 'employment_type', 'company_type', 'house_type', 'emi_scenario']

--- gender ---
gender
Male      237427
Female    158351
MALE        1865
M           1843
male        1815
F           1171
female      1165
FEMALE      1163
Name: count, dtype: int64

--- marital_status ---
marital_status
Married    307837
Single      96963
Name: count, dtype: int64

--- education ---
education
Graduate         181015
Post Graduate    100314
High School       60732
Professional      60335
NaN                2404
Name: count, dtype: int64

--- employment_type ---
employment_type
Private          283099
Government        81167
Self-employed     40534
Name: count, dtype: int64

--- company_type ---
company_type
Large Indian    121139
MNC             101409
Mid-size        101301
Startup          60706
Small            20245
Name: count, dtype: int64

--- house_type ---
house_type
Rented    161601
Own       142307
Family    100892
Name: count, d

In [17]:
# Standardize gender values

X['gender'] = X['gender'].str.strip().str.lower()

X['gender'] = X['gender'].replace({
    'm': 'male',
    'f': 'female'
})

print("Gender values after standardization:")
print(X['gender'].value_counts())

Gender values after standardization:
gender
male      242950
female    161850
Name: count, dtype: int64


C:\Users\kotgi\AppData\Local\Temp\ipykernel_25640\4134820487.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['gender'] = X['gender'].str.strip().str.lower()
C:\Users\kotgi\AppData\Local\Temp\ipykernel_25640\4134820487.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['gender'] = X['gender'].replace({


In [18]:
# Encode gender as numerical values

X['gender'] = X['gender'].map({
    'male': 1,
    'female': 0
})

print("Gender encoded successfully.")
print(X['gender'].value_counts())

Gender encoded successfully.
gender
1    242950
0    161850
Name: count, dtype: int64


C:\Users\kotgi\AppData\Local\Temp\ipykernel_25640\1277796895.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['gender'] = X['gender'].map({


In [19]:
# Encode marital status as numerical values

X['marital_status'] = X['marital_status'].map({
    'Married': 1,
    'Single': 0
})

print("Marital status encoded successfully.")
print(X['marital_status'].value_counts())

Marital status encoded successfully.
marital_status
1    307837
0     96963
Name: count, dtype: int64


C:\Users\kotgi\AppData\Local\Temp\ipykernel_25640\1492858506.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['marital_status'] = X['marital_status'].map({


In [20]:
# Check the education categories

print("Education categories:")
print(X['education'].value_counts(dropna=False))

Education categories:
education
Graduate         181015
Post Graduate    100314
High School       60732
Professional      60335
NaN                2404
Name: count, dtype: int64


In [21]:
# Fill missing education values with the most frequent category

X['education'] = X['education'].fillna(X['education'].mode()[0])

print("Missing education values:", X['education'].isnull().sum())
print("\nEducation categories after filling missing values:")
print(X['education'].value_counts())

Missing education values: 0

Education categories after filling missing values:
education
Graduate         183419
Post Graduate    100314
High School       60732
Professional      60335
Name: count, dtype: int64


C:\Users\kotgi\AppData\Local\Temp\ipykernel_25640\2256601135.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['education'] = X['education'].fillna(X['education'].mode()[0])


In [22]:
# One-hot encode the education column

education_encoded = pd.get_dummies(
    X['education'],
    prefix='education',
    dtype=int
)

# Add the encoded columns to X
X = pd.concat(
    [X.drop('education', axis=1), education_encoded],
    axis=1
)

print("Education encoded successfully.")
print("\nNew education columns:")
print(education_encoded.columns.tolist())

Education encoded successfully.

New education columns:
['education_Graduate', 'education_High School', 'education_Post Graduate', 'education_Professional']


In [24]:
# Check the categories and counts in employment_type

print("Employment type categories:")
print(X['employment_type'].value_counts(dropna=False))

Employment type categories:
employment_type
Private          283099
Government        81167
Self-employed     40534
Name: count, dtype: int64


In [25]:
# Check employment type categories

print(X['employment_type'].value_counts(dropna=False))

employment_type
Private          283099
Government        81167
Self-employed     40534
Name: count, dtype: int64


In [26]:
# One-hot encode employment type

employment_encoded = pd.get_dummies(
    X['employment_type'],
    prefix='employment_type',
    dummy_na=True,
    dtype=int
)

X = pd.concat(
    [X.drop('employment_type', axis=1), employment_encoded],
    axis=1
)

print("Employment type encoded successfully.")
print("New columns:")
print(employment_encoded.columns.tolist())

Employment type encoded successfully.
New columns:
['employment_type_Government', 'employment_type_Private', 'employment_type_Self-employed', 'employment_type_nan']


In [27]:
# Check company type categories

print(X['company_type'].value_counts(dropna=False))

company_type
Large Indian    121139
MNC             101409
Mid-size        101301
Startup          60706
Small            20245
Name: count, dtype: int64


In [28]:
# One-hot encode company type

company_encoded = pd.get_dummies(
    X['company_type'],
    prefix='company_type',
    dummy_na=True,
    dtype=int
)

X = pd.concat(
    [X.drop('company_type', axis=1), company_encoded],
    axis=1
)

print("Company type encoded successfully.")
print("New columns:")
print(company_encoded.columns.tolist())

Company type encoded successfully.
New columns:
['company_type_Large Indian', 'company_type_MNC', 'company_type_Mid-size', 'company_type_Small', 'company_type_Startup', 'company_type_nan']


In [29]:
# Check house type categories

print(X['house_type'].value_counts(dropna=False))

house_type
Rented    161601
Own       142307
Family    100892
Name: count, dtype: int64


In [30]:
# One-hot encode house type

house_encoded = pd.get_dummies(
    X['house_type'],
    prefix='house_type',
    dummy_na=True,
    dtype=int
)

X = pd.concat(
    [X.drop('house_type', axis=1), house_encoded],
    axis=1
)

print("House type encoded successfully.")
print("New columns:")
print(house_encoded.columns.tolist())

House type encoded successfully.
New columns:
['house_type_Family', 'house_type_Own', 'house_type_Rented', 'house_type_nan']


In [31]:
# Check EMI scenario categories

print(X['emi_scenario'].value_counts(dropna=False))

emi_scenario
Home Appliances EMI        80988
Personal Loan EMI          80980
E-commerce Shopping EMI    80948
Education EMI              80942
Vehicle EMI                80942
Name: count, dtype: int64


In [32]:
# One-hot encode EMI scenario

emi_scenario_encoded = pd.get_dummies(
    X['emi_scenario'],
    prefix='emi_scenario',
    dummy_na=True,
    dtype=int
)

X = pd.concat(
    [X.drop('emi_scenario', axis=1), emi_scenario_encoded],
    axis=1
)

print("EMI scenario encoded successfully.")
print("New columns:")
print(emi_scenario_encoded.columns.tolist())

EMI scenario encoded successfully.
New columns:
['emi_scenario_E-commerce Shopping EMI', 'emi_scenario_Education EMI', 'emi_scenario_Home Appliances EMI', 'emi_scenario_Personal Loan EMI', 'emi_scenario_Vehicle EMI', 'emi_scenario_nan']


In [33]:
# Final check for missing values

missing_values = X.isnull().sum()

print("Total missing values:", missing_values.sum())

if missing_values.sum() == 0:
    print("No missing values remain in the feature dataset.")
else:
    print("\nColumns with missing values:")
    print(missing_values[missing_values > 0])

Total missing values: 7197

Columns with missing values:
monthly_rent      2426
credit_score      2420
emergency_fund    2351
dtype: int64


In [34]:
# Fill remaining missing numerical values with their respective medians

remaining_numeric = [
    'monthly_rent',
    'credit_score',
    'emergency_fund'
]

for column in remaining_numeric:
    X[column] = X[column].fillna(X[column].median())

print("Remaining missing values handled successfully.")
print(X[remaining_numeric].isnull().sum())

Remaining missing values handled successfully.
monthly_rent      0
credit_score      0
emergency_fund    0
dtype: int64


In [35]:
# Final validation of the feature dataset

print("Feature dataset shape:", X.shape)

print("\nTotal missing values:", X.isnull().sum().sum())

print("\nRemaining object columns:")
print(X.select_dtypes(include='object').columns.tolist())

print("\nFeature data types:")
print(X.dtypes.value_counts())

Feature dataset shape: (404800, 44)

Total missing values: 0

Remaining object columns:
[]

Feature data types:
int64      30
float64    14
Name: count, dtype: int64


In [36]:
# Prepare classification and regression targets

y_classification = data['emi_eligibility']
y_regression = data['max_monthly_emi']

print("Classification target shape:", y_classification.shape)
print("Regression target shape:", y_regression.shape)

print("\nClassification classes:")
print(y_classification.value_counts())

Classification target shape: (404800,)
Regression target shape: (404800,)

Classification classes:
emi_eligibility
Not_Eligible    312868
Eligible         74444
High_Risk        17488
Name: count, dtype: int64


In [37]:
# Split data into training and testing sets

X_train, X_test, y_class_train, y_class_test = train_test_split(
    X,
    y_classification,
    test_size=0.20,
    random_state=42,
    stratify=y_classification
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_class_train.value_counts())

print("\nTesting target distribution:")
print(y_class_test.value_counts())

Training data shape: (323840, 44)
Testing data shape: (80960, 44)

Training target distribution:
emi_eligibility
Not_Eligible    250294
Eligible         59555
High_Risk        13991
Name: count, dtype: int64

Testing target distribution:
emi_eligibility
Not_Eligible    62574
Eligible        14889
High_Risk        3497
Name: count, dtype: int64


In [38]:
# Split data for the regression task

X_train_reg, X_test_reg, y_reg_train, y_reg_test = train_test_split(
    X,
    y_regression,
    test_size=0.20,
    random_state=42
)

print("Regression training data shape:", X_train_reg.shape)
print("Regression testing data shape:", X_test_reg.shape)

print("\nRegression training target shape:", y_reg_train.shape)
print("Regression testing target shape:", y_reg_test.shape)

Regression training data shape: (323840, 44)
Regression testing data shape: (80960, 44)

Regression training target shape: (323840,)
Regression testing target shape: (80960,)


In [39]:
# Save processed datasets for model training

X_train.to_csv('../data/X_train_classification.csv', index=False)
X_test.to_csv('../data/X_test_classification.csv', index=False)

y_class_train.to_csv('../data/y_train_classification.csv', index=False)
y_class_test.to_csv('../data/y_test_classification.csv', index=False)

X_train_reg.to_csv('../data/X_train_regression.csv', index=False)
X_test_reg.to_csv('../data/X_test_regression.csv', index=False)

y_reg_train.to_csv('../data/y_train_regression.csv', index=False)
y_reg_test.to_csv('../data/y_test_regression.csv', index=False)

print("All processed datasets saved successfully.")

All processed datasets saved successfully.
